# 02 — 定性可视化

从评测结果中随机抽取样本，展示成功与失败的典型案例（每个数据集 10–20 张）。

**依赖**：先运行完整评测，生成以下文件：
- `results/coco/predictions.json`
- `results/refcoco/refcoco_val_predictions.json`

In [1]:
import sys, json, random
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from PIL import Image

from src.utils.io import load_json
from src.analysis.visualize import visualize_ovd_sample, visualize_vg_sample

random.seed(0)
print('准备就绪')

准备就绪


## 1. OVD — COCO val2017 定性展示

In [ ]:
from pycocotools.coco import COCO

ANN_FILE = '../data/coco/annotations/instances_val2017.json'
IMG_DIR  = Path('../data/coco/val2017')
PRED_FILE = '../results/coco/predictions.json'

coco_gt = COCO(ANN_FILE)
coco_cats = {cat['id']: cat['name'] for cat in coco_gt.loadCats(coco_gt.getCatIds())}

# 读取预测，按 image_id 分组
predictions = load_json(PRED_FILE)
from collections import defaultdict
preds_by_img = defaultdict(list)
for p in predictions:
    preds_by_img[p['image_id']].append(p)

# 随机选取 12 张
sample_ids = random.sample(list(preds_by_img.keys()), min(12, len(preds_by_img)))

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, img_id in zip(axes.flat, sample_ids):
    img_info = coco_gt.loadImgs(img_id)[0]
    img_path = IMG_DIR / img_info['file_name']
    pil_img = Image.open(img_path).convert('RGB')

    preds = preds_by_img[img_id]
    # 只显示得分前 5
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)[:5]

    import numpy as np
    if preds:
        boxes = np.array([p['bbox'] for p in preds])
        # xywh → xyxy
        boxes[:, 2] += boxes[:, 0]; boxes[:, 3] += boxes[:, 1]
        scores = np.array([p['score'] for p in preds])
        labels = [coco_cats.get(p['category_id'], '?') for p in preds]
    else:
        boxes = np.zeros((0, 4)); scores = np.array([]); labels = []

    ax.imshow(pil_img)
    ax.set_title(f'img_id={img_id}', fontsize=7)
    ax.axis('off')
    import matplotlib.patches as mpatches
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        x1,y1,x2,y2 = box
        color = colors[i % len(colors)]
        ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,ec=color,fc='none'))
        ax.text(x1, y1-2, f'{label} {score:.2f}', fontsize=5, color='white',
                bbox={'fc': color, 'alpha': 0.7, 'pad': 0.5})

plt.suptitle('OVD — COCO val2017 随机可视化（top-5 检测）', fontsize=13)
plt.tight_layout()
plt.savefig('../results/coco/qualitative_ovd.png', dpi=100)
plt.show()

## 2. Visual Grounding — RefCOCO val 定性展示

In [ ]:
REFCOCO_PRED = '../results/refcoco/refcoco_val_predictions.json'
per_sample = load_json(REFCOCO_PRED)

hits   = [s for s in per_sample if s.get('hit')]
misses = [s for s in per_sample if not s.get('hit')]
print(f'命中：{len(hits)}，失败：{len(misses)}')

# 展示 5 个成功 + 5 个失败
show_hits   = random.sample(hits,   min(5, len(hits)))
show_misses = random.sample(misses, min(5, len(misses)))

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, sample in zip(axes[0], show_hits):
    pil_img = Image.open(sample['img_path']).convert('RGB')
    ax.imshow(pil_img)
    ax.set_title(f"HIT IoU={sample['iou']:.2f}\n{sample['expr'][:40]}", fontsize=7)
    ax.axis('off')
    import numpy as np, matplotlib.patches as mp
    if sample.get('pred_box'):
        x1,y1,x2,y2 = sample['pred_box']
        ax.add_patch(mp.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='blue',fc='none'))
    if sample.get('gt_box'):
        x1,y1,x2,y2 = sample['gt_box']
        ax.add_patch(mp.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='lime',fc='none',ls='--'))

for ax, sample in zip(axes[1], show_misses):
    pil_img = Image.open(sample['img_path']).convert('RGB')
    ax.imshow(pil_img)
    ax.set_title(f"MISS IoU={sample['iou']:.2f}\n{sample['expr'][:40]}", fontsize=7)
    ax.axis('off')
    if sample.get('pred_box'):
        x1,y1,x2,y2 = sample['pred_box']
        ax.add_patch(mp.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='red',fc='none'))
    if sample.get('gt_box'):
        x1,y1,x2,y2 = sample['gt_box']
        ax.add_patch(mp.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='lime',fc='none',ls='--'))

axes[0][0].set_ylabel('成功（蓝=pred, 绿=GT）', fontsize=9)
axes[1][0].set_ylabel('失败（红=pred, 绿=GT）', fontsize=9)

plt.suptitle('Visual Grounding — RefCOCO val 可视化', fontsize=13)
plt.tight_layout()
plt.savefig('../results/refcoco/qualitative_vg.png', dpi=100)
plt.show()

## 3. 指标汇总表

In [ ]:
import json
from IPython.display import display
import pandas as pd

# OVD 指标
coco_metrics_path = Path('../results/coco/metrics.json')
if coco_metrics_path.exists():
    coco_metrics = load_json(coco_metrics_path)
    print('=== OVD COCO val2017 ===')
    ovd_df = pd.DataFrame([{
        'mAP': f"{coco_metrics.get('mAP', 0)*100:.1f}",
        'AP50': f"{coco_metrics.get('AP50', 0)*100:.1f}",
        'AP75': f"{coco_metrics.get('AP75', 0)*100:.1f}",
        'AP_s': f"{coco_metrics.get('AP_s', 0)*100:.1f}",
        'AP_m': f"{coco_metrics.get('AP_m', 0)*100:.1f}",
        'AP_l': f"{coco_metrics.get('AP_l', 0)*100:.1f}",
    }])
    display(ovd_df)
else:
    print('（尚未生成 OVD 指标文件，请先运行 eval_coco.py）')

# VG 指标
vg_metrics_path = Path('../results/refcoco/metrics.json')
if vg_metrics_path.exists():
    vg_metrics = load_json(vg_metrics_path)
    print('\n=== VG RefCOCO/+/g ===')
    rows = []
    for ds, splits in vg_metrics.items():
        for split, m in splits.items():
            rows.append({'dataset': ds, 'split': split, 'Acc@0.5': f"{m.get('acc', 0)*100:.2f}",
                         'n_total': m.get('n_total', 0)})
    display(pd.DataFrame(rows))
else:
    print('（尚未生成 VG 指标文件，请先运行 eval_refcoco.py）')